# LLM Classifier Performance Validation
## Setup and sample annotation data
Prepare data and sample messages for manual annotation to validate the performance of the LLM classifier.


In [1]:
import os
import sys

sys.path.append("..")
os.makedirs("results", exist_ok=True)
from intent_classification.categories import VALID_CATEGORIES
import pandas as pd

df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)


SAMPLE_N = 20
RANDOM_SEED = 42
OUTPUT_PATH = "results/annotation_sample_all_categories.csv"

# ── Context lookup: (sha, index_in_chat) -> truncated_content of prior message
context_lookup = (
    df_classifications[["sha", "index_in_chat", "truncated_content"]]
    .drop_duplicates(subset=["sha", "index_in_chat"])
    .set_index(["sha", "index_in_chat"])["truncated_content"]
)

all_subcategories = [sub for subs in VALID_CATEGORIES.values() for sub in subs]

# ── Value counts for reference ────────────────────────────────────────────────
vc = (
    df_classifications[df_classifications["sub_category"].isin(all_subcategories)]
    .drop_duplicates(subset=["sha", "index_in_chat", "sub_category"])["sub_category"]
    .value_counts()
)

# ── Sample and build records ──────────────────────────────────────────────────
records = []

row_count = 0
for sub_category in all_subcategories:
    target_msgs = df_classifications[
        df_classifications["sub_category"] == sub_category
    ][["sha", "index_in_chat"]].drop_duplicates()

    n_available = len(target_msgs)
    if n_available == 0:
        print(f"[SKIP] {sub_category}: no messages found")
        continue

    sampled = target_msgs.sample(min(SAMPLE_N, n_available), random_state=RANDOM_SEED)
    if n_available < SAMPLE_N:
        print(f"[WARN] {sub_category}: only {n_available} available, sampling all")

    for _, (sha, idx) in sampled[["sha", "index_in_chat"]].iterrows():
        msg_rows = df_classifications[
            (df_classifications["sha"] == sha)
            & (df_classifications["index_in_chat"] == idx)
        ].sort_values("label_rank")

        row_count += 1
        records.append(
            {
                "id": row_count,
                "prev_msg (context)": context_lookup.get((sha, idx - 1), ""),
                "sample_category": sub_category,  # which category this row was sampled for
                "repository_full_name": msg_rows["repository_full_name"].iloc[0],
                "sha": sha,
                "msg_index": idx,
                "msg_content": msg_rows["content"].iloc[0],
                "num_labels": len(msg_rows),
                "labels": "\n".join(msg_rows["sub_category"].tolist()),
                "reasonings": "\n".join(msg_rows["reasoning"].tolist()),
            }
        )

df_out = pd.DataFrame(records)
df_out = df_out.replace(
    "3.1 Planning & Decision Consultation", "3.1 Planning & Consultation", regex=True
)
df_out.to_csv(OUTPUT_PATH, index=False, quoting=1)

print(
    f"\nSaved {len(df_out)} rows ({len(all_subcategories)} categories × up to {SAMPLE_N}) to {OUTPUT_PATH}"
)

[SKIP] 8.1 Others: no messages found

Saved 400 rows (21 categories × up to 20) to results/annotation_sample_all_categories.csv


## Classify messages with Claude
Run classification using Claude API and save results.

In [2]:
import json
import time
import anthropic
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

MODEL = "claude-sonnet-4-6"
SAMPLE_CSV = "results/annotation_sample_all_categories.csv"
CODEBOOK_PATH = "../intent_classification/codebook.txt"
OUTPUT_JSONL = "results/claude_classifications.jsonl"

# ── Structured output tool ─────────────────────────────────────────────────────
CLASSIFY_TOOL = {
    "name": "classify_message",
    "description": "Classify a user message with one or more behavioral intent labels.",
    "input_schema": {
        "type": "object",
        "properties": {
            "labels": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "reasoning": {"type": "string"},
                        "main_category": {"type": "string"},
                        "sub_category": {"type": "string"},
                    },
                    "required": ["reasoning", "main_category", "sub_category"],
                },
                "minItems": 1,
            }
        },
        "required": ["labels"],
    },
}

# ── Load inputs ────────────────────────────────────────────────────────────────
codebook = Path(CODEBOOK_PATH).read_text()
df = pd.read_csv(SAMPLE_CSV)
msgs = df.drop_duplicates(subset=["id"])[
    ["id", "prev_msg (context)", "msg_content"]
].reset_index(drop=True)

# ── Resume: skip already-done ids ─────────────────────────────────────────────
done_ids = set()
if Path(OUTPUT_JSONL).exists():
    with open(OUTPUT_JSONL) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])

remaining = msgs[~msgs["id"].isin(done_ids)]
print(f"Resuming — {len(done_ids)} done, {len(remaining)} remaining")

# ── Classify ───────────────────────────────────────────────────────────────────
client = anthropic.Anthropic()

with open(OUTPUT_JSONL, "a") as out:
    for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc="Classifying"):
        user_prompt = (
            f"[prev_user]: {row['prev_msg (context)'] or ''}\n\n"
            f"[current]: {row['msg_content']}"
        )

        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=[
                {
                    "type": "text",
                    "text": codebook,
                    "cache_control": {
                        "type": "ephemeral"
                    },  # cache the codebook across all calls
                }
            ],
            tools=[CLASSIFY_TOOL],
            tool_choice={"type": "tool", "name": "classify_message"},
            messages=[{"role": "user", "content": user_prompt}],
        )

        tool_input = next(b.input for b in response.content if b.type == "tool_use")

        record = {
            "id": row["id"],
            "model": MODEL,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "cache_creation_input_tokens": getattr(
                response.usage, "cache_creation_input_tokens", 0
            ),
            "cache_read_input_tokens": getattr(
                response.usage, "cache_read_input_tokens", 0
            ),
            "labels": tool_input["labels"],
        }
        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        out.flush()

        time.sleep(0.2)

print(f"\nDone — results in {OUTPUT_JSONL}")

/Users/ningzhi_tang/Documents/VSCodeProjects/empirical-conversational-programming/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resuming — 400 done, 0 remaining


Classifying: 0it [00:00, ?it/s]


Done — results in results/claude_classifications.jsonl


## Cost calculation
Calculate token usage and cost for Claude classification.

In [3]:
import json
import pandas as pd

JSONL_PATH = "results/claude_classifications.jsonl"

# Sonnet 4.6 pricing (USD per million tokens)
PRICE = {
    "input": 3.00 / 1e6,
    "cache_write": 3.75 / 1e6,  # 1.25x base
    "cache_read": 0.30 / 1e6,  # 0.10x base
    "output": 15.00 / 1e6,
}

records = [json.loads(l) for l in open(JSONL_PATH)]
df = pd.DataFrame(records)

totals = {
    "input_tokens": df["input_tokens"].sum(),
    "cache_write_tokens": df["cache_creation_input_tokens"].sum(),
    "cache_read_tokens": df["cache_read_input_tokens"].sum(),
    "output_tokens": df["output_tokens"].sum(),
}

costs = {k: totals[k] * PRICE[k.replace("_tokens", "")] for k in totals}

summary = pd.DataFrame({"tokens": totals, "cost_usd": costs})
summary.loc["total"] = summary.sum()
print(summary.to_string(float_format="%.4f"))
print(f"\nTotal cost: ${summary.loc['total', 'cost_usd']:.4f}")

                         tokens  cost_usd
input_tokens        303281.0000    0.9098
cache_write_tokens    4619.0000    0.0173
cache_read_tokens  1842981.0000    0.5529
output_tokens        59527.0000    0.8929
total              2210408.0000    2.3730

Total cost: $2.3730


## Agreement analysis
Pairwise inter-rater agreement (Cohen's κ) for any two label columns.

In [4]:
from sklearn.metrics import cohen_kappa_score
from intent_classification.categories import VALID_CATEGORIES


# ── Taxonomy setup ─────────────────────────────────────────────────────────────


def _build_subcategories() -> list[str]:
    subs = [
        sub
        for main, sub_list in VALID_CATEGORIES.items()
        for sub in sub_list
        if main != "8. Others"
    ]
    return [
        (
            s
            if s != "3.1 Planning & Decision Consultation"
            else "3.1 Planning & Consultation"
        )
        for s in subs
    ]


SUBCATEGORIES = _build_subcategories()
MAIN_CATEGORIES = [m for m in VALID_CATEGORIES if m != "8. Others"]


# ── Core compute ───────────────────────────────────────────────────────────────


def compute_metrics(
    df: pd.DataFrame,
    col_a: str,
    col_b: str,
) -> pd.DataFrame:
    """
    Per-subcategory binary Cohen's κ and P/R/F1.

    col_a = reference (gold), col_b = predicted  (for P/R/F1 direction).
    κ is symmetric so direction doesn't matter for that metric.
    """
    results = []
    for main_cat, subs in VALID_CATEGORIES.items():
        if main_cat == "8. Others":
            continue
        for sub in subs:
            sub_key = (
                "3.1 Planning & Consultation"
                if sub == "3.1 Planning & Decision Consultation"
                else sub
            )
            y_a = df[col_a].apply(lambda ls: int(sub_key in ls))
            y_b = df[col_b].apply(lambda ls: int(sub_key in ls))

            tp = int(((y_a == 1) & (y_b == 1)).sum())
            fp = int(((y_a == 0) & (y_b == 1)).sum())
            fn = int(((y_a == 1) & (y_b == 0)).sum())

            precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
            recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
            f1 = (
                2 * precision * recall / (precision + recall)
                if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
                else float("nan")
            )

            # positive (specific) agreement — symmetric
            pos_agree = (
                2 * tp / (y_a.sum() + y_b.sum())
                if (y_a.sum() + y_b.sum()) > 0
                else float("nan")
            )

            n_pos_a = int(y_a.sum())
            n_pos_b = int(y_b.sum())
            n_agree = int((y_a == y_b).sum())

            if y_a.nunique() == 1 or y_b.nunique() == 1:
                kappa = float("nan")
            else:
                kappa = cohen_kappa_score(y_a, y_b)

            results.append(
                {
                    "main_category": main_cat,
                    "sub_category": sub_key,
                    "kappa": kappa,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    "pos_agree": pos_agree,
                    "exact_agree": n_agree,
                    "n_pos_a": n_pos_a,
                    "n_pos_b": n_pos_b,
                }
            )
    return pd.DataFrame(results)


def compute_exact_set_match(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    """Fraction of rows where the two label sets are identical."""
    return df.apply(lambda r: set(r[col_a]) == set(r[col_b]), axis=1).mean()


def compute_mean_jaccard(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    """Mean instance-level Jaccard similarity."""

    def jaccard(r):
        a, b = set(r[col_a]), set(r[col_b])
        if not a and not b:
            return 1.0
        return len(a & b) / len(a | b)

    return df.apply(jaccard, axis=1).mean()


# ── Pretty-print ───────────────────────────────────────────────────────────────


def _fmt(val) -> str:
    return f"{val:.3f}" if pd.notna(val) else "  n/a"


def print_report(
    df_res: pd.DataFrame,
    col_a: str,
    col_b: str,
    exact_set: float,
    jaccard: float,
    n_rows: int,
) -> None:
    """Print full per-label table with κ + P/R/F1 + summary."""
    header = f"═══  {col_a}  vs  {col_b}  ({n_rows} messages)  ═══"
    print(f"\n{'═' * len(header)}")
    print(header)
    print(f"{'═' * len(header)}")
    print(f"  (P/R/F1: {col_a} = reference, {col_b} = predicted)\n")

    a_short = col_a.split("_", 1)[-1] if "_" in col_a else col_a
    b_short = col_b.split("_", 1)[-1] if "_" in col_b else col_b

    print(
        f"{'Sub-category':<38} {'κ':>6}  {'P':>6}  {'R':>6}  {'F1':>6}  "
        f"{'P_ag':>6}  {a_short + '+':>5}  {b_short + '+':>5}"
    )
    print("─" * 95)

    prev_main = None
    for _, row in df_res.iterrows():
        if row["main_category"] != prev_main:
            print(f"\n{row['main_category']}")
            prev_main = row["main_category"]
        print(
            f"  {row['sub_category']:<36} "
            f"{_fmt(row['kappa']):>6}  "
            f"{_fmt(row['precision']):>6}  "
            f"{_fmt(row['recall']):>6}  "
            f"{_fmt(row['f1']):>6}  "
            f"{_fmt(row['pos_agree']):>6}  "
            f"{row['n_pos_a']:>5}  {row['n_pos_b']:>5}"
        )

    # ── Summary ────────────────────────────────────────────────────────────
    print(f"\n{'─' * 95}")

    valid_k = df_res["kappa"].dropna()
    valid_p = df_res["precision"].dropna()
    valid_r = df_res["recall"].dropna()
    valid_f = df_res["f1"].dropna()

    print(f"  Macro-avg κ :  {valid_k.mean():.3f}")
    print(f"  Macro-avg P :  {valid_p.mean():.3f}")
    print(f"  Macro-avg R :  {valid_r.mean():.3f}")
    print(f"  Macro-avg F1:  {valid_f.mean():.3f}")

    print(f"\n  Per main category:")
    print(f"  {'Category':<35} {'κ':>6}  {'P':>6}  {'R':>6}  {'F1':>6}  {'n':>3}")
    print(f"  {'─' * 70}")
    for main, grp in df_res.groupby("main_category"):
        vk = grp["kappa"].dropna()
        vp = grp["precision"].dropna()
        vr = grp["recall"].dropna()
        vf = grp["f1"].dropna()
        if len(vf):
            print(
                f"  {main:<35} {vk.mean():.3f}  {vp.mean():.3f}  "
                f"{vr.mean():.3f}  {vf.mean():.3f}  {len(vf):>3}"
            )

    print(f"\n  Exact set match:    {exact_set:.1%}")
    print(f"  Mean Jaccard:       {jaccard:.3f}")


# ── Convenience wrapper ────────────────────────────────────────────────────────


def compare(df: pd.DataFrame, col_a: str, col_b: str) -> None:
    """
    Compute + print κ and F1 between two label columns.
    col_a = reference for P/R/F1.
    """
    df_res = compute_metrics(df, col_a, col_b)
    exact_set = compute_exact_set_match(df, col_a, col_b)
    jaccard = compute_mean_jaccard(df, col_a, col_b)
    print_report(df_res, col_a, col_b, exact_set, jaccard, len(df))

In [5]:
df_labels = pd.read_csv("all_annotated_labels.csv")

In [6]:
compare(df_labels, "labels_a1", "labels_a2")


══════════════════════════════════════════════════
═══  labels_a1  vs  labels_a2  (400 messages)  ═══
══════════════════════════════════════════════════
  (P/R/F1: labels_a1 = reference, labels_a2 = predicted)

Sub-category                                κ       P       R      F1    P_ag    a1+    a2+
───────────────────────────────────────────────────────────────────────────────────────────────

1. Code Authoring
  1.1 New Implementation                0.682   0.704   0.704   0.704   0.704     27     27
  1.2 Iterative Modification            0.661   0.771   0.649   0.705   0.705     57     48
  1.3 Alignment Correction              0.465   0.368   0.700   0.483   0.483     10     19

2. Failure Reporting
  2.1 Log Paste                         0.690   0.607   0.850   0.708   0.708     20     28
  2.2 Symptom Description               0.592   0.609   0.667   0.636   0.636     42     46
  2.3 Error Persistence                 0.778   0.731   0.864   0.792   0.792     22     26

3. Inq

In [7]:
compare(df_labels, "labels_consensus", "labels_gpt")


══════════════════════════════════════════════════════════
═══  labels_consensus  vs  labels_gpt  (400 messages)  ═══
══════════════════════════════════════════════════════════
  (P/R/F1: labels_consensus = reference, labels_gpt = predicted)

Sub-category                                κ       P       R      F1    P_ag  consensus+   gpt+
───────────────────────────────────────────────────────────────────────────────────────────────

1. Code Authoring
  1.1 New Implementation                0.863   0.889   0.857   0.873   0.873     28     27
  1.2 Iterative Modification            0.777   0.759   0.863   0.807   0.807     51     58
  1.3 Alignment Correction              0.507   0.444   0.667   0.533   0.533     18     27

2. Failure Reporting
  2.1 Log Paste                         0.981   0.966   1.000   0.982   0.982     28     29
  2.2 Symptom Description               0.783   0.905   0.731   0.809   0.809     52     42
  2.3 Error Persistence                 0.877   0.885   0.885 